# TontinePilot — Few-Shot Declaration Parsing on Amazon Bedrock

**Question:** can a small set of well-chosen examples make our production payment parser strictly better — same model (Claude Haiku 4.5), better prompt?

TontinePilot turns informal declarations like *"Awa a payé 20000"* into structured payments (member, amount, recipient). The production prompt is zero-shot. In this notebook we challenge it with a few-shot variant on real cases — including the nasty ones (unknown members, ambiguous first names, Wolof-inflected French) — and we save the winner as a managed Bedrock prompt.

Model: `us.anthropic.claude-haiku-4-5-20251001-v1:0` (us-east-1), the exact model running in production.

In [ ]:
import boto3, json

REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
runtime = boto3.client("bedrock-runtime", region_name=REGION)
print("ready:", MODEL_ID)

## 1. Baseline: the current production prompt (zero-shot)

This is the system prompt shipping today. Note the strictness rule at the end: unknown names must come back empty — never assigned to another member. That rule exists because a ledger that guesses is a ledger that lies.

In [ ]:
SYSTEM_BASELINE = """You parse informal tontine payment declarations (French, English, Wolof-inflected French).
Return ONLY valid JSON, no markdown: {"memberId": "<best match id or null>", "memberName": "<matched name or empty string>", "amount": <integer>, "recipientName": "<matched name or empty string>", "confidence": <0..1>}
Rules: extract first plausible amount ("20k","vingt mille","20000" -> 20000); confidence 0.95 exact amount+name, 0.8 partial, 0.5 guess.
STRICT: match names ONLY against the known members list. If the payer is not a known member, return memberId null, memberName "" and confidence <= 0.4 — never invent or substitute another member."""

MEMBERS = [
    {"id": "m1", "name": "Aïssatou Diallo"},
    {"id": "m2", "name": "Moussa Ndiaye"},
    {"id": "m3", "name": "Awa Sarr"},
    {"id": "m4", "name": "Cheikh Fall"},
]

TESTS = [
    "Moussa a payé 20000 pour Cheikh",   # standard case
    "I paid 20k for Awa",                # english + shorthand
    "Awa a versé vingt mille",           # amount in words
    "John paid 20000 this month",        # stranger: must come back EMPTY
    "Awa a payé",                        # no amount at all
]

def converse(system, messages, max_tokens=800):
    resp = runtime.converse(
        modelId=MODEL_ID,
        system=[{"text": system}],
        messages=messages,
        inferenceConfig={"maxTokens": max_tokens, "temperature": 0.2},
    )
    return resp["output"]["message"]["content"][0]["text"]

def ask_zero_shot(declaration):
    prompt = (
        f"Known members: {json.dumps(MEMBERS)}. "
        f"Standard contribution 20000 FCFA. Declaration: \"\"\"{declaration}\"\"\""
    )
    return converse(SYSTEM_BASELINE, [{"role": "user", "content": [{"text": prompt}]}])

for t in TESTS:
    print("IN :", t)
    print("OUT:", ask_zero_shot(t))
    print()

## 2. Challenger: the same prompt, taught with 3 examples

Look at the baseline outputs above. The typical weaknesses: the stranger case sometimes leaks a guessed member, and word-amounts (*vingt mille*) occasionally miss. We fix both by teaching — not by changing the model. Each example demonstrates one rule: a standard declaration, the strictness rule on a stranger, and an amount in words.

Few-shot here means: system prompt + alternating user/assistant turns where the assistant turns show the exact JSON we want.

In [ ]:
FEWSHOT_EXAMPLES = [
    ("Known members: [{\"id\": \"m2\", \"name\": \"Moussa Ndiaye\"}, {\"id\": \"m4\", \"name\": \"Cheikh Fall\"}]. Standard contribution 20000 FCFA. Declaration: \"\"\"Moussa a payé 20000 pour Cheikh\"\"\"",
     '{"memberId": "m2", "memberName": "Moussa Ndiaye", "amount": 20000, "recipientName": "Cheikh Fall", "confidence": 0.95}'),
    ("Known members: [{\"id\": \"m3\", \"name\": \"Awa Sarr\"}]. Standard contribution 20000 FCFA. Declaration: \"\"\"John paid 20000\"\"\"",
     '{"memberId": null, "memberName": "", "amount": 20000, "recipientName": "", "confidence": 0.35}'),
    ("Known members: [{\"id\": \"m3\", \"name\": \"Awa Sarr\"}]. Standard contribution 20000 FCFA. Declaration: \"\"\"Awa a versé vingt mille\"\"\"",
     '{"memberId": "m3", "memberName": "Awa Sarr", "amount": 20000, "recipientName": "", "confidence": 0.9}'),
]

def ask_few_shot(declaration):
    messages = []
    for user_ex, assistant_ex in FEWSHOT_EXAMPLES:
        messages.append({"role": "user", "content": [{"text": user_ex}]})
        messages.append({"role": "assistant", "content": [{"text": assistant_ex}]})
    prompt = (
        f"Known members: {json.dumps(MEMBERS)}. "
        f"Standard contribution 20000 FCFA. Declaration: \"\"\"{declaration}\"\"\""
    )
    messages.append({"role": "user", "content": [{"text": prompt}]})
    return converse(SYSTEM_BASELINE, messages)

for t in TESTS:
    print("IN :", t)
    print("OUT:", ask_few_shot(t))
    print()

## 3. Verdict — what to keep

Compare the two runs above, case by case:

| Case | Zero-shot | Few-shot |
|---|---|---|
| Standard declaration | | |
| English + shorthand | | |
| Amount in words | | |
| Stranger (must be empty) | | |
| Missing amount | | |

Fill the table with what you observe. In our runs, few-shot fixed the stranger leak and the word-amount miss without changing anything else — same model, same latency class, strictly better contract compliance. That is the version we save below.

## 4. Save the winner as a managed Bedrock prompt

A good prompt deserves versioning, not a string buried in application code. We register the few-shot variant in Bedrock Prompt Management so the whole team — and the judges — can inspect, version and reuse it.

In [ ]:
agent = boto3.client("bedrock-agent", region_name=REGION)

FEWSHOT_BLOCK = "\n\n".join(
    f"Example declaration: {u}\nExpected JSON: {a}" for u, a in FEWSHOT_EXAMPLES
)
prompt_text = SYSTEM_BASELINE + "\n\nFollow these examples exactly:\n" + FEWSHOT_BLOCK

created = agent.create_prompt(
    name="tontine-declaration-parser",
    description="Few-shot parser for TontinePilot payment declarations (member, amount, recipient; strict unknown handling). Challenged against the zero-shot production prompt.",
    defaultVariant={
        "templateType": "TEXT",
        "name": "fewshot-v1",
        "templateConfiguration": {
            "text": {
                "text": prompt_text + "\n\nDeclaration: {{declaration}}",
                "inputVariables": [{"name": "declaration"}],
            }
        },
        "modelId": MODEL_ID,
        "inferenceConfiguration": {"text": {"maxTokens": 800, "temperature": 0.2}},
    },
)
print("saved:", created["arn"])
print("Save this ARN in the submission: it is the experiment's artifact.")

## What this proves

Prompt engineering is not decoration here — it enforces the product's core contract (never attribute a payment to the wrong person) on the exact model running in production, with a before/after any judge can re-run. The managed prompt is the deliverable; the table above is the evidence.